# Download COCO Datasets

#### Images — ~778 MB
`!wget -c http://images.cocodataset.org/zips/val2017.zip`

#### Annotations — ~241 MB (contains train + val; you need the val one)
`!wget -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip`

`!unzip val2017.zip`
`!unzip annotations_trainval2017.zip`

# Testing on YOLO World vs COCO dataset for object detection and comparison of results.

In [20]:
import os, torch
from pycocotools.coco import COCO
from ultralytics import YOLOWorld
from torchmetrics.detection import MeanAveragePrecision

In [21]:
ANN     = "coco/annotations/instances_val2017.json"
IMG_DIR = "coco/val2017"
PERSON  = 1                      # COCO category id for person


In [22]:
coco    = COCO(ANN)
img_ids = sorted(coco.getImgIds())[:100]


loading annotations into memory...
Done (t=1.77s)
creating index...
index created!


In [23]:
model = YOLOWorld("yolov8s-world.pt").to("cuda" if torch.cuda.is_available() else "cpu")     # first run downloads ~50 MB
model.set_classes(["person"])

records = []                              # keep raw dets for offline sweeping


In [24]:
for iid in img_ids:
    info = coco.loadImgs(iid)[0]
    path = os.path.join(IMG_DIR, info["file_name"])

    anns = coco.loadAnns(coco.getAnnIds(imgIds=iid, catIds=[PERSON], iscrowd=False))
    gt = torch.tensor( 
        [[x, y, x + w, y + h] for x, y, w, h in (a["bbox"] for a in anns)],
        dtype=torch.float32,
    ).reshape(-1, 4)
                                                               
    r = model.predict(path, conf=0.001, max_det=300, verbose=False)[0]
    records.append({"boxes": r.boxes.xyxy.cpu(), "scores": r.boxes.conf.cpu(), "gt": gt})

print(f"{len(records)} images, "
      f"{sum(len(x['gt']) for x in records)} ground-truth people, "
      f"{sum(len(x['boxes']) for x in records)} raw detections")


100 images, 283 ground-truth people, 3664 raw detections


In [25]:
def evaluate(records, thresh):
    metric = MeanAveragePrecision(iou_type="bbox")
    for r in records:
        keep = r["scores"] >= thresh
        metric.update(
            [{"boxes": r["boxes"][keep],
              "scores": r["scores"][keep],
              "labels": torch.zeros(int(keep.sum()), dtype=torch.long)}],
            [{"boxes": r["gt"],
              "labels": torch.zeros(len(r["gt"]), dtype=torch.long)}],
        )
    m = metric.compute()
    return float(m["map_50"]), float(m["map"])


In [26]:
for t in [0.01, 0.05, 0.10, 0.25, 0.40, 0.60]:
    ap50, ap = evaluate(records, t)
    print(f"conf {t:.2f}   mAP50 {ap50:.3f}   mAP50-95 {ap:.3f}")


conf 0.01   mAP50 0.713   mAP50-95 0.506
conf 0.05   mAP50 0.692   mAP50-95 0.493
conf 0.10   mAP50 0.651   mAP50-95 0.469
conf 0.25   mAP50 0.552   mAP50-95 0.420
conf 0.40   mAP50 0.483   mAP50-95 0.373
conf 0.60   mAP50 0.373   mAP50-95 0.308


In [27]:
import torch
from torchvision.ops import box_iou

def pr_at(records, thresh, iou_thr=0.5):
    tp = fp = fn = 0
    for r in records:
        keep   = r["scores"] >= thresh
        boxes  = r["boxes"][keep]
        scores = r["scores"][keep]
        gt     = r["gt"]

        boxes = boxes[scores.argsort(descending=True)]     # best first
        matched = torch.zeros(len(gt), dtype=torch.bool)

        for b in boxes:
            if len(gt) == 0:
                fp += 1
                continue
            ious = box_iou(b.unsqueeze(0), gt)[0].clone()
            ious[matched] = 0                              # each person counted once
            best = int(ious.argmax())
            if ious[best] >= iou_thr:
                matched[best] = True
                tp += 1
            else:
                fp += 1
        fn += int((~matched).sum())

    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return prec, rec, f1                                     


In [28]:
for t in [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]:
    p, r, f = pr_at(records, t)
    print(f"conf {t:.2f}   P {p:.3f}   R {r:.3f}   F1 {f:.3f}")


conf 0.01   P 0.229   R 0.859   F1 0.361
conf 0.05   P 0.458   R 0.799   F1 0.582
conf 0.10   P 0.610   R 0.714   F1 0.658
conf 0.15   P 0.699   R 0.664   F1 0.681
conf 0.20   P 0.748   R 0.618   F1 0.677
conf 0.25   P 0.788   R 0.580   F1 0.668
conf 0.30   P 0.831   R 0.555   F1 0.665
conf 0.40   P 0.904   R 0.498   F1 0.642
conf 0.50   P 0.931   R 0.431   F1 0.589
conf 0.60   P 0.947   R 0.378   F1 0.540


In [18]:
import numpy as np

CLASSES = ["person", "chair", "backpack", "umbrella",
           "scissors", "toaster", "toothbrush", "hair drier"]

name_to_id = {c["name"]: c["id"] for c in coco.loadCats(coco.getCatIds())}

def build_records(class_name, n_pos=100, n_neg=100):
    cid     = name_to_id[class_name]
    pos_all = set(coco.getImgIds(catIds=[cid]))
    pos     = sorted(pos_all)[:n_pos]
    neg     = [i for i in sorted(coco.getImgIds()) if i not in pos_all][:n_neg]

    model.set_classes([class_name])
    
    recs = []
    for iid in pos + neg:
        info = coco.loadImgs(iid)[0]
        path = os.path.join(IMG_DIR, info["file_name"])
        anns = coco.loadAnns(coco.getAnnIds(imgIds=iid, catIds=[cid], iscrowd=False))
        gt = torch.tensor(
            [[x, y, x + w, y + h] for x, y, w, h in (a["bbox"] for a in anns)],
            dtype=torch.float32,
        ).reshape(-1, 4)
        r = model.predict(path, conf=0.001, max_det=300, verbose=False)[0]
        recs.append({"boxes": r.boxes.xyxy.cpu(), "scores": r.boxes.conf.cpu(), "gt": gt})
    return recs


In [19]:
THRESHOLDS = [round(t, 2) for t in np.arange(0.01, 0.65, 0.02)]

for c in CLASSES:
    recs = build_records(c)
    best = max(((t, *pr_at(recs, t)) for t in THRESHOLDS), key=lambda x: x[3])
    n_gt = sum(len(r["gt"]) for r in recs)
    print(f"{c:12s} best conf {best[0]:.2f}  P {best[1]:.3f}  R {best[2]:.3f}  "
          f"F1 {best[3]:.3f}  (gt={n_gt})")


person       best conf 0.33  P 0.847  R 0.582  F1 0.690  (gt=466)
chair        best conf 0.25  P 0.570  R 0.357  F1 0.439  (gt=308)
backpack     best conf 0.13  P 0.530  R 0.360  F1 0.429  (gt=172)
umbrella     best conf 0.25  P 0.672  R 0.614  F1 0.641  (gt=207)
scissors     best conf 0.27  P 0.824  R 0.389  F1 0.528  (gt=36)
toaster      best conf 0.27  P 1.000  R 0.778  F1 0.875  (gt=9)
toothbrush   best conf 0.25  P 0.762  R 0.281  F1 0.410  (gt=57)
hair drier   best conf 0.01  P 0.188  R 0.273  F1 0.222  (gt=11)


In [29]:
def prevalence_test(class_name, n_pos=100, neg_counts=(0, 50, 100, 200, 400)):
    cid     = name_to_id[class_name]
    pos_all = set(coco.getImgIds(catIds=[cid]))
    pos     = sorted(pos_all)[:n_pos]
    neg_pool = [i for i in sorted(coco.getImgIds()) if i not in pos_all]

    model.set_classes([class_name])
    cache = {}
    def rec_for(iid):
        if iid not in cache:
            info = coco.loadImgs(iid)[0]
            path = os.path.join(IMG_DIR, info["file_name"])
            anns = coco.loadAnns(coco.getAnnIds(imgIds=iid, catIds=[cid], iscrowd=False))
            gt = torch.tensor(
                [[x, y, x + w, y + h] for x, y, w, h in (a["bbox"] for a in anns)],
                dtype=torch.float32).reshape(-1, 4)
            r = model.predict(path, conf=0.001, max_det=300, verbose=False)[0]
            cache[iid] = {"boxes": r.boxes.xyxy.cpu(), "scores": r.boxes.conf.cpu(), "gt": gt}
        return cache[iid]

    for n_neg in neg_counts:
        recs = [rec_for(i) for i in pos + neg_pool[:n_neg]]
        best = max(((t, *pr_at(recs, t)) for t in THRESHOLDS), key=lambda x: x[3])
        prev = n_pos / (n_pos + n_neg)
        print(f"prevalence {prev:5.1%}  ({n_pos}p/{n_neg}n)  "
              f"best conf {best[0]:.2f}  P {best[1]:.3f}  R {best[2]:.3f}  F1 {best[3]:.3f}")

prevalence_test("person")

prevalence 100.0%  (100p/0n)  best conf 0.23  P 0.768  R 0.631  F1 0.693
prevalence 66.7%  (100p/50n)  best conf 0.23  P 0.760  R 0.631  F1 0.689
prevalence 50.0%  (100p/100n)  best conf 0.29  P 0.801  R 0.603  F1 0.688
prevalence 33.3%  (100p/200n)  best conf 0.29  P 0.792  R 0.603  F1 0.685
prevalence 20.0%  (100p/400n)  best conf 0.29  P 0.768  R 0.603  F1 0.675


# Trying with trained YOLO V8n

In [30]:
from ultralytics import YOLO

trained = YOLO("yolov8n.pt")
yolo_idx = {v: k for k, v in trained.names.items()}     # name -> YOLO index 0..79

def build_records_trained(class_name, n_pos=100, n_neg=100):
    cid     = name_to_id[class_name]        # COCO id  -> ground truth
    yidx    = yolo_idx[class_name]          # YOLO idx -> predictions
    pos_all = set(coco.getImgIds(catIds=[cid]))
    pos     = sorted(pos_all)[:n_pos]
    neg     = [i for i in sorted(coco.getImgIds()) if i not in pos_all][:n_neg]

    recs = []
    for iid in pos + neg:
        info = coco.loadImgs(iid)[0]
        path = os.path.join(IMG_DIR, info["file_name"])
        anns = coco.loadAnns(coco.getAnnIds(imgIds=iid, catIds=[cid], iscrowd=False))
        gt = torch.tensor(
            [[x, y, x + w, y + h] for x, y, w, h in (a["bbox"] for a in anns)],
            dtype=torch.float32).reshape(-1, 4)

        r = trained.predict(path, conf=0.001, max_det=300, verbose=False)[0]
        m = r.boxes.cls == yidx                          # keep only this class
        recs.append({"boxes": r.boxes.xyxy[m].cpu(),
                     "scores": r.boxes.conf[m].cpu(),
                     "gt": gt})
    return recs


In [31]:
CLASSES = ["person", "chair", "backpack", "umbrella", "scissors", "toothbrush"]

print(f"{'class':12s} {'prompted':>9s} {'trained':>9s} {'gap':>8s}")
for c in CLASSES:
    b_ow = max(((t, *pr_at(build_records(c),         t)) for t in THRESHOLDS), key=lambda x: x[3])
    b_tr = max(((t, *pr_at(build_records_trained(c), t)) for t in THRESHOLDS), key=lambda x: x[3])
    print(f"{c:12s} {b_ow[3]:9.3f} {b_tr[3]:9.3f} {b_tr[3]-b_ow[3]:+8.3f}")


class         prompted   trained      gap
person           0.688     0.726   +0.038
chair            0.439     0.445   +0.006
backpack         0.429     0.310   -0.119
umbrella         0.641     0.631   -0.010
scissors         0.528     0.449   -0.079
toothbrush       0.410     0.358   -0.052


In [32]:
trained_yolo_v8s = YOLO("yolov8s.pt")
CLASSES = ["person", "chair", "backpack", "umbrella", "scissors", "toothbrush"]
def build_records_trained(class_name, n_pos=100, n_neg=100):
    cid     = name_to_id[class_name]        # COCO id  -> ground truth
    yidx    = yolo_idx[class_name]          # YOLO idx -> predictions
    pos_all = set(coco.getImgIds(catIds=[cid]))
    pos     = sorted(pos_all)[:n_pos]
    neg     = [i for i in sorted(coco.getImgIds()) if i not in pos_all][:n_neg]

    recs = []
    for iid in pos + neg:
        info = coco.loadImgs(iid)[0]
        path = os.path.join(IMG_DIR, info["file_name"])
        anns = coco.loadAnns(coco.getAnnIds(imgIds=iid, catIds=[cid], iscrowd=False))
        gt = torch.tensor(
            [[x, y, x + w, y + h] for x, y, w, h in (a["bbox"] for a in anns)],
            dtype=torch.float32).reshape(-1, 4)

        r = trained_yolo_v8s.predict(path, conf=0.001, max_det=300, verbose=False)[0]
        m = r.boxes.cls == yidx                          # keep only this class
        recs.append({"boxes": r.boxes.xyxy[m].cpu(),
                     "scores": r.boxes.conf[m].cpu(),
                     "gt": gt})
    return recs
print(f"{'class':12s} {'prompted':>9s} {'trained_yolo_v8s':>9s} {'gap':>8s}")
for c in CLASSES:
    b_ow = max(((t, *pr_at(build_records(c),         t)) for t in THRESHOLDS), key=lambda x: x[3])
    b_tr = max(((t, *pr_at(build_records_trained(c), t)) for t in THRESHOLDS), key=lambda x: x[3])
    print(f"{c:12s} {b_ow[3]:9.3f} {b_tr[3]:9.3f} {b_tr[3]-b_ow[3]:+8.3f}")


class         prompted trained_yolo_v8s      gap
person           0.688     0.743   +0.056
chair            0.439     0.547   +0.108
backpack         0.429     0.403   -0.026
umbrella         0.641     0.745   +0.103
scissors         0.528     0.549   +0.021
toothbrush       0.410     0.491   +0.081
